# Exercise 2 express — the sextortion study with the modern `ext` API

The previous notebooks talk to the REST API through the generated low-level client: every call needs a `Configuration`, an `ApiClient` context manager, the right `*Api` class, and manual CSV plumbing for bulk requests.

The `graphsense.ext` module is the **modern take on the API** — a hand-written convenience layer on top of the generated client that removes most of that boilerplate. For new code we recommend starting here:

* **One object** — `GraphSense` holds the connection, the API key, and a default currency.
* **Bundled lookups** — `lookup_address` / `lookup_cluster` fetch the main object plus auxiliary data (tags, cluster, …) in parallel and return them as a single `Bundle`.
* **Simple bulk** — `gs.bulk(operation, keys)` wraps the `/bulk` endpoint; CSV comes back as a string ready for pandas.
* **Pagination handled** — `gs.tags_for(address)` walks `next_page` tokens for you.
* **Full raw access** — every non-deprecated generated API stays reachable under `gs.raw.<group>` for anything the facade doesn't cover.

We demonstrate this by re-running the sextortion study from Exercise 2 in a fraction of the code.

## Setup

One import, one object. The API key and host can also come from the environment (`GRAPHSENSE_API_KEY` / `GRAPHSENSE_HOST`), in which case `GraphSense()` needs no arguments at all.

In [1]:
import io
import json

import pandas as pd

from graphsense.ext import GraphSense

config = json.load(open('config.json'))['graphsense']
gs = GraphSense(api_key=config['api_key'], host=config['host'], currency='btc')

A quick connectivity check:

In [2]:
[c.name for c in gs.statistics().currencies]

['btc', 'bch', 'ltc', 'zec', 'eth', 'trx']

## One-call address dossiers

With the low-level client, fetching an address together with its tags and its cluster takes three endpoint calls. `lookup_address` does all of them in parallel and returns a `Bundle`:

In [3]:
bundle = gs.lookup_address('1Archive1n2C579dMsAu3iC6tWzuQJz8dN',
                           with_tags=True, with_cluster=True)

cluster = bundle.cluster.actual_instance
print(f"Address {bundle.data.address}")
print(f"  received:  {bundle.data.total_received.fiat_values[0].value} EUR")
print(f"  cluster:   {cluster.cluster} ({cluster.no_addresses} addresses)")
print(f"  tags:      {sorted({t.label for t in bundle.tags.address_tags})}")

Address 1Archive1n2C579dMsAu3iC6tWzuQJz8dN
  received:  7773598.0 EUR
  cluster:   2647118 (126 addresses)
  tags:      ['Dark Web', 'Individual', 'Internet Archive']


## The sextortion study, condensed

We load the same 244 perpetrator addresses as in Exercise 2 and answer the study's questions, one or two lines each.

In [4]:
addressDF = pd.read_csv('./data/sextortion_addresses.csv')
addressDF.columns = ['address']
len(addressDF)

244

### Q1: How many USD did the input addresses receive in total?

`gs.bulk` posts all addresses to the `/bulk` endpoint in one go; with `format='csv'` the result is a CSV string we feed straight into pandas.

In [5]:
addrDF = pd.read_csv(io.StringIO(gs.bulk('get_address', addressDF['address'], format='csv')))
print(f"{len(addrDF)} addresses received {addrDF['total_received_usd'].sum():,.2f} USD")

244 addresses received 885,304.59 USD


### Q2: To how many clusters can we map these addresses?

In [6]:
clusters = addrDF['cluster'].drop_duplicates()
print(f"They map to {len(clusters)} distinct clusters")

They map to 95 distinct clusters


### Q3 & Q4: How many addresses do these clusters control, and how many USD did they receive?

As in Exercise 2 we drop service/super-clusters (more than 10K addresses) before aggregating.

In [7]:
clusterDF = pd.read_csv(io.StringIO(gs.bulk('get_cluster', clusters,
                                            key_field='cluster', format='csv')))
focusDF = clusterDF[clusterDF['no_addresses'] < 10e3]
print(f"{len(focusDF)} focus clusters control {focusDF['no_addresses'].sum()} addresses " +
      f"and received {focusDF['total_received_usd'].sum():,.2f} USD")

93 focus clusters control 559 addresses and received 1,512,140.02 USD


### Q5: Where did the money come from, and where did it go to?

`gs.bulk` covers the common case of a single key list. The neighbors operation needs extra body parameters (`direction`, `include_labels`), so we drop down one level to the raw generated client — it is always available under `gs.raw`:

In [ ]:
def cluster_neighbors(cluster_ids, direction):
    return pd.read_csv(gs.raw.bulk.bulk_csv('btc', 'list_cluster_neighbors',
                                            body={'cluster': list(cluster_ids),
                                                  'direction': direction,
                                                  'include_labels': True},
                                            num_pages=2, _preload_content=False))

sendingDF = cluster_neighbors(focusDF['cluster'], 'in')
receivingDF = cluster_neighbors(focusDF['cluster'], 'out')
len(sendingDF), len(receivingDF)

We keep only tagged counterparties outside our focus set. The senders are dominated by exchange custodial wallets — the victims' money enters from there:

In [ ]:
known_sendersDF = sendingDF[~sendingDF['cluster_cluster'].isin(sendingDF['_request_cluster'])] \
    .dropna(subset=['cluster_best_address_tag_label'])
known_sendersDF.groupby('cluster_best_address_tag_label').size() \
    .sort_values(ascending=False).head(10)

And on the outgoing side, the clusters that received funds from the perpetrator:

In [ ]:
known_receiversDF = receivingDF[~receivingDF['cluster_cluster'].isin(receivingDF['_request_cluster'])] \
    .dropna(subset=['cluster_best_address_tag_label'])
known_receiversDF[['_request_cluster', 'cluster_cluster', 'cluster_best_address_tag_label']] \
    .rename(columns={'_request_cluster': 'src_cluster',
                     'cluster_cluster': 'dst_cluster',
                     'cluster_best_address_tag_label': 'label'})

## Bonus: paginated tags without the loop

`tags_for` walks the `next_page` tokens of `list_tags_by_address` transparently and returns plain dicts:

In [ ]:
tags = gs.tags_for('1Archive1n2C579dMsAu3iC6tWzuQJz8dN')
[(t['label'], t['confidence']) for t in tags['address_tags'][:5]]

When we are done, we close the underlying client (or use `GraphSense` as a context manager):

In [ ]:
gs.close()

## Recap

The whole Exercise 2 analysis took five short code cells:

| Task | Low-level client | Modern `ext` API |
| --- | --- | --- |
| Connection | `Configuration` + `ApiClient` per call | one `GraphSense` object |
| Address + tags + cluster | 3 sequential calls | `lookup_address(..., with_tags=True, with_cluster=True)` |
| Bulk requests | build body dict, `_preload_content=False` | `gs.bulk(operation, keys, format='csv')` |
| Tag pagination | manual `next_page` loop | `gs.tags_for(address)` |
| Anything else | — | still there, under `gs.raw.<group>` |